In [1]:
# === Setup ===
from pathlib import Path

MODELS = Path("../../models")
BINARY_CHECKPOINT = MODELS / "04_02_therapistqa_binary" / "model"
PR_10_WAY_CLASS_CHECKPOINT = MODELS / "02_03_roberta_patternreframe_softlabels_alpha" / "alpha_0.3" / "model"

RESULTS_DIR      = Path("./results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
# Hand-curated deployment-realistic examples.
# Each example is a single thought as would appear in the
# "automatic thoughts" field of the application after being prompted by a
# situation. They are NOT taken from any test set.

EXAMPLES = [
    # ===== Obviously distorted (one per category) =====
    # All-or-nothing
    ("distorted", "If I don't get this promotion, my whole career is a failure.",
     "All-or-nothing thinking"),
    # Catastrophizing
    ("distorted", "I made one mistake in the presentation and now everyone will think I'm incompetent.",
     "Catastrophizing"),
    # Discounting the positive
    ("distorted", "She said she liked my work but she was probably just being polite.",
     "Discounting the positive"),
    # Fortune-telling
    ("distorted", "I just know this date is going to go badly.",
     "Fortune-telling"),
    # Mind reading
    ("distorted", "He didn't text me back, so he must be angry with me.",
     "Mind reading"),
    # Labeling
    ("distorted", "I forgot to send the email. I'm such an idiot.",
     "Labeling"),
    # Mental filter
    ("distorted", "Everyone clapped except for one person who looked bored. The whole talk was a disaster.",
     "Mental filtering"),
    # Overgeneralization
    ("distorted", "I bombed this job interview. I'm never going to find work.",
     "Overgeneralization"),
    # Personalization
    ("distorted", "My partner is in a bad mood. It must be something I did.",
     "Personalization"),
    # Should statements
    ("distorted", "I should be further along in my career by now.",
     "Should statements"),

    # ===== non-distorted =====
    ("nondistorted", "I should call my mother this weekend.", "No Distortion"),
    ("nondistorted", "It looks like it might rain later.", "No Distortion"),
    ("nondistorted", "I have a meeting at 3pm tomorrow.", "No Distortion"),
    ("nondistorted", "The coffee tastes good today.", "No Distortion"),
    ("nondistorted", "I want to read more this year.", "No Distortion"),
    ("nondistorted", "My back hurts after sitting all day.", "No Distortion"),
    ("nondistorted", "I'm feeling tired and I think I'll go to bed early.", "No Distortion"),
    ("nondistorted", "The deadline is on Friday so I need to finish the report by Thursday.", "No Distortion"),
    ("nondistorted", "I disagreed with what she said but I understand her point.", "No Distortion"),
    ("nondistorted", "I made some progress today, though there's still more to do.", "No Distortion"),

    # ===== Ambiguous / borderline =====
    # Could be a fact or a distortion depending on intent
    ("ambiguous", "He didn't say hi to me at the meeting.", "uncertain"),
    # Genuine concern vs catastrophizing — depends on context
    ("ambiguous", "I'm worried I won't have enough money for retirement.", "uncertain"),
    # Realistic expectation vs all-or-nothing
    ("ambiguous", "If the funding doesn't come through, the project is over.", "uncertain"),
    # Reasonable inference vs mind reading
    ("ambiguous", "My boss has been avoiding eye contact this week.", "uncertain"),
    # Self-criticism that may be deserved
    ("ambiguous", "I shouldn't have lost my temper with him.", "uncertain"),
    # Reasonable concern vs fortune-telling
    ("ambiguous", "Given the market right now, jobs in my field are going to be hard to find.", "uncertain"),
    # Mixed positive and negative
    ("ambiguous", "I did well overall but I keep thinking about that one bad answer.", "uncertain"),
    # Reasonable doubt
    ("ambiguous", "I don't think I'm qualified for this role.", "uncertain"),
    # Self-blame that may or may not be appropriate
    ("ambiguous", "The team missed the deadline and I was the lead.", "uncertain"),
    # General malaise
    ("ambiguous", "Something doesn't feel right but I can't put my finger on what.", "uncertain"),
]

print(f"Total examples: {len(EXAMPLES)}")
print(f"  Obviously distorted:    {sum(1 for x in EXAMPLES if x[0] == 'distorted')}")
print(f"  Obviously non-distorted: {sum(1 for x in EXAMPLES if x[0] == 'nondistorted')}")
print(f"  Ambiguous / borderline:  {sum(1 for x in EXAMPLES if x[0] == 'ambiguous')}")

Total examples: 30
  Obviously distorted:    10
  Obviously non-distorted: 10
  Ambiguous / borderline:  10


In [3]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.nn.functional import softmax

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# TQA-trained binary detector (stage 1)
print(f"\nLoading binary detector from {BINARY_CHECKPOINT}")
tok_b = AutoTokenizer.from_pretrained(BINARY_CHECKPOINT)
mdl_b = AutoModelForSequenceClassification.from_pretrained(BINARY_CHECKPOINT)
mdl_b.to(device).eval()
cfg_b = [mdl_b.config.id2label[i] for i in sorted(mdl_b.config.id2label, key=int)]
print(f"  Binary detector labels: {cfg_b}")
assert cfg_b == ["No Distortion", "Distorted"], f"Unexpected binary labels: {cfg_b}"

# PR 10-way type classifier (stage 2, soft-label α=0.3)
print(f"\nLoading 10-way type classifier from {PR_10_WAY_CLASS_CHECKPOINT}")
tok_t = AutoTokenizer.from_pretrained(PR_10_WAY_CLASS_CHECKPOINT)
mdl_t = AutoModelForSequenceClassification.from_pretrained(PR_10_WAY_CLASS_CHECKPOINT)
mdl_t.to(device).eval()
cfg_t = [mdl_t.config.id2label[i] for i in sorted(mdl_t.config.id2label, key=int)]
print(f"  Type classifier labels (10-way): {cfg_t}")
assert len(cfg_t) == 10, f"Expected 10 distortion classes, found {len(cfg_t)}"

/Users/ivagoluza/Documents/cognitive-distortioins-ml/dataset_preparation/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Device: cpu

Loading binary detector from ../../models/04_02_therapistqa_binary/model
  Binary detector labels: ['No Distortion', 'Distorted']

Loading 10-way type classifier from ../../models/02_03_roberta_patternreframe_softlabels_alpha/alpha_0.3/model
  Type classifier labels (10-way): ['Black-and-white or polarized thinking / All or nothing thinking', 'Catastrophizing', 'Discounting the positive', 'Jumping to conclusions: Fortune-telling', 'Jumping to conclusions: mind reading', 'Labeling and mislabeling', 'Mental filtering', 'Overgeneralization', 'Personalization', 'Should statements']


In [4]:
MAX_LEN = 512


def predict_one(text, tokenizer, model):
    """Return the full probability distribution for a single input."""
    enc = tokenizer(text, truncation=True, max_length=MAX_LEN,
                    return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**enc).logits
    return softmax(logits, dim=-1).cpu().numpy()[0]


# Run both models on all 30 examples
binary_probs = []   # prob(distorted) from the binary detector
type_argmax  = []   # the type classifier's argmax label (distortion category)
type_max_prob = []  # the type classifier's confidence

for intent, text, _ in EXAMPLES:
    # Binary detector
    pb = predict_one(text, tok_b, mdl_b)
    binary_probs.append(float(pb[1]))  # prob(distorted)

    # Type classifier
    pt = predict_one(text, tok_t, mdl_t)
    type_argmax.append(cfg_t[int(np.argmax(pt))])
    type_max_prob.append(float(np.max(pt)))

print(f"Done. {len(EXAMPLES)} examples processed.")

Done. 30 examples processed.


In [5]:
import pandas as pd

# Pipeline rule: if binary says ND (prob_distorted < 0.5), output "No Distortion";
# otherwise, output the type classifier's argmax category.
pipeline_preds = []
binary_calls   = []
for i in range(len(EXAMPLES)):
    is_distorted = binary_probs[i] >= 0.5
    binary_calls.append("Distorted" if is_distorted else "No Distortion")
    pipeline_preds.append(type_argmax[i] if is_distorted else "No Distortion")

# Map the casual expected-label names to the long-form PR labels for matching
EXPECTED_TO_LONG = {
    "All-or-nothing thinking":  "Black-and-white or polarized thinking / All or nothing thinking",
    "Catastrophizing":          "Catastrophizing",
    "Discounting the positive": "Discounting the positive",
    "Fortune-telling":          "Jumping to conclusions: Fortune-telling",
    "Mind reading":             "Jumping to conclusions: mind reading",
    "Labeling":                 "Labeling and mislabeling",
    "Mental filtering":         "Mental filtering",
    "Overgeneralization":       "Overgeneralization",
    "Personalization":          "Personalization",
    "Should statements":        "Should statements",
    "No Distortion":            "No Distortion",
    "uncertain":                "uncertain",
}

# Short labels for compact display
SHORT_LABEL = {
    "No Distortion":                                                    "ND",
    "Black-and-white or polarized thinking / All or nothing thinking":  "All-or-nothing",
    "Catastrophizing":                                                  "Catastrophizing",
    "Discounting the positive":                                         "Discounting +",
    "Jumping to conclusions: Fortune-telling":                          "Fortune-telling",
    "Jumping to conclusions: mind reading":                             "Mind reading",
    "Labeling and mislabeling":                                         "Labeling",
    "Mental filtering":                                                 "Mental filter",
    "Overgeneralization":                                               "Overgeneralization",
    "Personalization":                                                  "Personalization",
    "Should statements":                                                "Should",
    "uncertain":                                                        "(ambiguous)",
}

rows = []
for i, (intent, text, expected) in enumerate(EXAMPLES):
    long_expected = EXPECTED_TO_LONG[expected]
    if expected == "uncertain":
        agreement = "ambiguous"
    elif pipeline_preds[i] == long_expected:
        agreement = "✓ match"
    else:
        agreement = "✗ different"

    rows.append({
        "Intent":              intent,
        "Thought":             text if len(text) <= 70 else text[:67] + "...",
        "Expected":            SHORT_LABEL[long_expected],
        "Binary call":         binary_calls[i],
        "Bin prob(D)":         f"{binary_probs[i]:.2f}",
        "Pipeline pred":       SHORT_LABEL[pipeline_preds[i]],
        "Type argmax":         SHORT_LABEL[type_argmax[i]],
        "Type max-prob":       f"{type_max_prob[i]:.2f}",
        "Agreement":           agreement,
    })

results = pd.DataFrame(rows)
print(results.to_string(index=False))
results.to_csv(RESULTS_DIR / "deployment_examples_pipeline.csv", index=False)
print(f"\nSaved to {RESULTS_DIR / 'deployment_examples_pipeline.csv'}")

      Intent                                                                Thought           Expected   Binary call Bin prob(D)      Pipeline pred        Type argmax Type max-prob   Agreement
   distorted           If I don't get this promotion, my whole career is a failure.     All-or-nothing     Distorted        0.96     All-or-nothing     All-or-nothing          0.86     ✓ match
   distorted I made one mistake in the presentation and now everyone will think ...    Catastrophizing     Distorted        0.98 Overgeneralization Overgeneralization          0.42 ✗ different
   distorted     She said she liked my work but she was probably just being polite.      Discounting + No Distortion        0.05                 ND      Discounting +          0.90 ✗ different
   distorted                            I just know this date is going to go badly.    Fortune-telling     Distorted        0.97    Fortune-telling    Fortune-telling          0.74     ✓ match
   distorted                   He d

In [6]:
print("=== Summary by intent ===\n")

for intent in ["distorted", "nondistorted", "ambiguous"]:
    sub_idx = [i for i, ex in enumerate(EXAMPLES) if ex[0] == intent]
    n = len(sub_idx)

    if intent == "distorted":
        # How often does the pipeline match the expected category?
        matches = sum(
            1 for i in sub_idx
            if pipeline_preds[i] == EXPECTED_TO_LONG[EXAMPLES[i][2]]
        )
        # How often does the binary stage correctly call them distorted?
        bin_caught = sum(1 for i in sub_idx if binary_probs[i] >= 0.5)

        print(f"DISTORTED ({n} examples):")
        print(f"  Binary stage flagged as distorted: {bin_caught}/{n}")
        print(f"  Pipeline matched expected category: {matches}/{n}")
        # How many were missed at the binary stage (false negatives)?
        missed = n - bin_caught
        if missed:
            print(f"  ⚠ {missed} distorted thoughts missed by binary stage:")
            for i in sub_idx:
                if binary_probs[i] < 0.5:
                    print(f"     - prob(D)={binary_probs[i]:.2f}: \"{EXAMPLES[i][1][:60]}...\"")
        print()

    elif intent == "nondistorted":
        # How often does the pipeline correctly say ND?
        correct_nd = sum(1 for i in sub_idx if pipeline_preds[i] == "No Distortion")
        bin_correct_nd = sum(1 for i in sub_idx if binary_probs[i] < 0.5)

        print(f"NONDISTORTED ({n} examples):")
        print(f"  Binary stage correctly said 'No Distortion': {bin_correct_nd}/{n}")
        print(f"  Pipeline correctly output 'No Distortion':   {correct_nd}/{n}")
        # On the ones where the binary stage failed, what did the pipeline output?
        fp_idx = [i for i in sub_idx if binary_probs[i] >= 0.5]
        if fp_idx:
            print(f"  ⚠ {len(fp_idx)} neutral thoughts misclassified as distorted:")
            for i in fp_idx:
                print(f"     - prob(D)={binary_probs[i]:.2f}, type→{SHORT_LABEL[type_argmax[i]]}: "
                      f"\"{EXAMPLES[i][1][:50]}...\"")
        print()

    else:  # ambiguous
        nd_pipeline = sum(1 for i in sub_idx if pipeline_preds[i] == "No Distortion")
        avg_bin_prob = np.mean([binary_probs[i] for i in sub_idx])
        avg_type_max = np.mean([type_max_prob[i] for i in sub_idx])

        print(f"AMBIGUOUS ({n} examples):")
        print(f"  Pipeline output 'No Distortion': {nd_pipeline}/{n}")
        print(f"  Mean binary prob(distorted):     {avg_bin_prob:.3f}")
        print(f"  Mean type-classifier max-prob:   {avg_type_max:.3f}")
        print()

# Confidence tier breakdown of binary probabilities
print("=== Binary stage confidence distribution ===")
for lo, hi, name in [
    (0.0, 0.3, "high confidence ND"),
    (0.3, 0.5, "leaning ND but uncertain"),
    (0.5, 0.7, "leaning distorted but uncertain"),
    (0.7, 1.0, "high confidence distorted"),
]:
    count = sum(1 for p in binary_probs if lo <= p < hi)
    print(f"  prob(D) ∈ [{lo:.1f}, {hi:.1f}) — {name}: {count} examples")

=== Summary by intent ===

DISTORTED (10 examples):
  Binary stage flagged as distorted: 8/10
  Pipeline matched expected category: 6/10
  ⚠ 2 distorted thoughts missed by binary stage:
     - prob(D)=0.05: "She said she liked my work but she was probably just being p..."
     - prob(D)=0.03: "I should be further along in my career by now...."

NONDISTORTED (10 examples):
  Binary stage correctly said 'No Distortion': 10/10
  Pipeline correctly output 'No Distortion':   10/10

AMBIGUOUS (10 examples):
  Pipeline output 'No Distortion': 6/10
  Mean binary prob(distorted):     0.405
  Mean type-classifier max-prob:   0.487

=== Binary stage confidence distribution ===
  prob(D) ∈ [0.0, 0.3) — high confidence ND: 17 examples
  prob(D) ∈ [0.3, 0.5) — leaning ND but uncertain: 1 examples
  prob(D) ∈ [0.5, 0.7) — leaning distorted but uncertain: 3 examples
  prob(D) ∈ [0.7, 1.0) — high confidence distorted: 9 examples
